# Patent inventor countries — `countries`, `n_countries`, first-inventor country

One row per US **utility** patent that has at least one disambiguated inventor, keyed on
`patent_id` like every other file in `PatentView/output/`. The PatentView twin of the
`countries` / `n_countries` columns in `Dimensions/output/paper_author.parquet` and of
`OpenAlex/notebook/paper_author_country.ipynb`.

## Raw data
PatentsView **Granted** bulk `.tsv.zip` (2026-05-21 snapshot):
```
g_patent.tsv.zip                    patent_id, patent_type, patent_date   -> utility filter
g_inventor_disambiguated.tsv.zip    patent_id, inventor_sequence, inventor_id, location_id
g_assignee_disambiguated.tsv.zip    patent_id, assignee_sequence, assignee_id, location_id
g_location_disambiguated.tsv.zip    location_id, disambig_country (ISO2)   -> the country lookup
```
An inventor's country is the `disambig_country` of the `location_id` PatentsView attached to
that inventor **on that patent** (the address printed on the grant), not the inventor's current
or most frequent location. The location table is fully populated: 100,452 locations, none
without a country.

## Two traps
1. **Namibia.** Its ISO2 code is `NA`. `pd.read_csv` turns that into NaN by default, so every
   Namibian inventor would silently become "unlocated". Every read below uses
   `keep_default_na=False, na_values=['']`.
2. **One inventor, several rows.** `patent_metadata.ipynb` de-duplicates inventors on
   `(patent_id, inventor_id)`; so does this notebook, keeping the lowest sequence, so that
   `n_inventors` here equals `len(inventor_list.split(';'))` there. That identity is asserted in
   the verification section.

## Output — `PatentView/output/patent_inventor_country.parquet`
| column | |
|---|---|
| `patent_id` | utility patent with ≥ 1 inventor row |
| `n_inventors` | distinct inventors on the patent |
| `n_located` | inventors whose location resolves to a country |
| `countries` | distinct ISO2 codes over all located inventors, **sorted**, `;`-joined; null if none |
| `n_countries` | `len(countries.split(';'))`, 0 when null |
| `country_inventor_counts` | inventors per country, `US:3;JP:1`, ordered by count desc then code |
| `first_inventor_country` | country of the lowest-sequence inventor; null if *that* inventor is unlocated (not promoted to the next located one) |
| `is_international` | `n_countries > 1` |
| `assignee_countries` | same construction over the patent's disambiguated assignees; kept beside the inventor columns because "patent country" in the literature usually means the assignee's |

Environment hooks for a smoke test: `NB_ROW_LIMIT` (read only the first N inventor / assignee
rows) and `NB_OUT_DIR` (write somewhere other than `PatentView/output`).

In [ ]:
import os, sys, gc, time
import numpy as np, pandas as pd
import duckdb
sys.path.insert(0, '/project/jevans/Dawoon/Science of Science/PatentView')
import pv_common as pv

OUT_DIR   = os.environ.get('NB_OUT_DIR', pv.OUT)
ROW_LIMIT = int(os.environ['NB_ROW_LIMIT']) if os.environ.get('NB_ROW_LIMIT') else None
OUT_FP    = f'{OUT_DIR}/patent_inventor_country.parquet'
META_FP   = pv.out('patent_metadata.parquet')          # sibling, for the cross-check only
os.makedirs(OUT_DIR, exist_ok=True)
pv.preflight('patent_inventor_country')
if ROW_LIMIT:
    print(f'\n*** SMOKE TEST: first {ROW_LIMIT:,} inventor/assignee rows only -> {OUT_FP}')

# Namibia is 'NA'. Never let pandas decide what a missing value looks like in these files.
READ = dict(sep='\t', dtype=str, keep_default_na=False, na_values=[''])

## 1. Utility patent universe (same rule as `patent_metadata`)

In [ ]:
%%time
gp = pd.read_csv(pv.granted('g_patent.tsv.zip'), usecols=['patent_id', 'patent_type', 'patent_date'], **READ)
gp = gp[gp['patent_type'] == 'utility']
gp['grant_year'] = pd.to_datetime(gp['patent_date'], errors='coerce').dt.year
years = gp.dropna(subset=['grant_year']).set_index('patent_id')['grant_year'].astype(int)
util  = set(years.index)
print(f'utility patents with a grant date: {len(util):,}')
del gp; gc.collect()

## 2. Location → ISO2 country

In [ ]:
%%time
loc = pd.read_csv(pv.granted('g_location_disambiguated.tsv.zip'), usecols=['location_id', 'disambig_country'], **READ)
loc['disambig_country'] = loc['disambig_country'].str.strip().str.upper()
bad = loc[~loc['disambig_country'].fillna('').str.fullmatch(r'[A-Z]{2}')]
print(f'{len(loc):,} locations, {loc.disambig_country.nunique():,} distinct countries, '
      f'{loc.disambig_country.isna().sum():,} without country, {len(bad):,} not ISO2-shaped')
if len(bad): display(bad.head())
assert loc['location_id'].is_unique
country_of = loc.dropna(subset=['disambig_country']).set_index('location_id')['disambig_country']
print('Namibia present as NA:', (country_of == 'NA').sum(), 'locations')

## 3. Inventors and assignees, one row per (patent, person) with a country

`inventor_sequence` orders the inventors as printed on the grant; the lowest one is the
"first inventor". A person appearing twice on one patent keeps the lowest sequence and the
country attached to that row.

In [ ]:
%%time
def people(fn, seq_col, id_col):
    t0 = time.time()
    d = pd.read_csv(pv.granted(fn), usecols=['patent_id', seq_col, id_col, 'location_id'], nrows=ROW_LIMIT, **READ)
    n_raw = len(d)
    d = d[d['patent_id'].isin(util)].dropna(subset=[id_col])
    d['seq'] = pd.to_numeric(d[seq_col], errors='coerce')
    d['country'] = d['location_id'].map(country_of)          # NaN where no / unknown location
    d = (d.sort_values(['patent_id', 'seq']).drop_duplicates(['patent_id', id_col])
          .rename(columns={id_col: 'person_id'})[['patent_id', 'person_id', 'seq', 'country']]
          .reset_index(drop=True))
    print(f'  {fn:36s} {n_raw:>12,} rows -> {len(d):>12,} (patent, person) on utility patents, '
          f'{d.country.notna().mean()*100:5.1f}% located   [{time.time()-t0:.0f}s]')
    return d

inv = people('g_inventor_disambiguated.tsv.zip', 'inventor_sequence', 'inventor_id')
asg = people('g_assignee_disambiguated.tsv.zip', 'assignee_sequence', 'assignee_id')
print(f'patents with >=1 inventor: {inv.patent_id.nunique():,}   with >=1 assignee: {asg.patent_id.nunique():,}')

## 4. Aggregate per patent

DuckDB over the two frames: `list_distinct` + `list_sort` for the country set, a second
grouping for the per-country inventor counts, and `list(country ORDER BY seq)[1]` for the first
inventor's country — which is NULL when the first inventor is unlocated, on purpose.

In [ ]:
%%time
con = duckdb.connect()
con.execute("SET preserve_insertion_order=false")
con.register('inv', inv); con.register('asg', asg)
t0 = time.time()
con.execute(f"""
COPY (
  WITH p AS (
    SELECT patent_id,
           count(*)                                          AS n_inventors,
           count(country)                                    AS n_located,
           list_sort(list_distinct(list(country) FILTER (WHERE country IS NOT NULL))) AS clist,
           list(country ORDER BY seq, person_id)[1]          AS first_inventor_country
    FROM inv GROUP BY patent_id),
  c AS (
    SELECT patent_id, string_agg(country || ':' || n, ';' ORDER BY n DESC, country) AS country_inventor_counts
    FROM (SELECT patent_id, country, count(*) AS n FROM inv WHERE country IS NOT NULL GROUP BY 1, 2)
    GROUP BY patent_id),
  a AS (
    SELECT patent_id,
           array_to_string(list_sort(list_distinct(list(country) FILTER (WHERE country IS NOT NULL))), ';') AS assignee_countries
    FROM asg GROUP BY patent_id)
  SELECT p.patent_id,
         p.n_inventors::INTEGER                         AS n_inventors,
         p.n_located::INTEGER                           AS n_located,
         nullif(array_to_string(coalesce(p.clist, []), ';'), '') AS countries,
         coalesce(len(p.clist), 0)::INTEGER             AS n_countries,
         c.country_inventor_counts,
         p.first_inventor_country,
         coalesce(len(p.clist), 0) > 1                  AS is_international,
         nullif(a.assignee_countries, '')               AS assignee_countries
  FROM p LEFT JOIN c USING (patent_id) LEFT JOIN a USING (patent_id)
  ORDER BY p.patent_id
) TO '{OUT_FP}.tmp' (FORMAT PARQUET, COMPRESSION ZSTD, ROW_GROUP_SIZE 1000000)""")
os.replace(OUT_FP + '.tmp', OUT_FP)
n = con.execute(f"SELECT count(*) FROM read_parquet('{OUT_FP}')").fetchone()[0]
print(f'WROTE {OUT_FP}  ({os.path.getsize(OUT_FP)/1e6:.1f} MB, {n:,} patents) in {time.time()-t0:.0f}s')

## 5. Verification

1. `patent_id` unique; `n_located ≤ n_inventors`; `n_countries == len(countries.split(';'))`.
2. `country_inventor_counts` sums to `n_located` and names exactly the codes in `countries`
   (a different route to the same numbers — `count(country)` versus a `GROUP BY country`).
3. `first_inventor_country`, when not null, is a member of `countries`; `is_international`
   agrees with `n_countries`.
4. `n_inventors` equals the length of `patent_metadata.inventor_list` for every shared patent —
   both de-duplicate on `(patent_id, inventor_id)`, so any gap is a real defect in one of them.
   The list is split where the **next id starts** (`;fl:` or `;` + a 20-character hash id),
   not at every `;`: 36 patents carry an inventor id with a `;` of its own — an HTML entity
   (`fl:iv_ln:jadri&cacute;-1`) or a mangled ß (`fl:jo_ln:ha;feld-1`) — which a plain split
   counts as a separator.
5. Coverage: located share by grant decade, and the most frequent countries.

In [ ]:
%%time
chk = con.execute(f"""
WITH s AS (SELECT * FROM read_parquet('{OUT_FP}')),
     x AS (SELECT *, CASE WHEN countries IS NULL THEN [] ELSE str_split(countries, ';') END AS parts,
                     CASE WHEN country_inventor_counts IS NULL THEN []
                          ELSE list_transform(str_split(country_inventor_counts, ';'), z -> str_split(z, ':')) END AS cc
           FROM s)
SELECT count(*)                                                                     AS patents,
       count(*) - count(DISTINCT patent_id)                                         AS dup_patent_id,
       sum(n_inventors)                                                             AS inventor_slots,
       sum(n_located)                                                               AS located_slots,
       count(*) FILTER (WHERE n_located > n_inventors)                              AS located_gt_inventors,
       count(*) FILTER (WHERE n_countries <> len(parts))                            AS n_countries_mismatch,
       count(*) FILTER (WHERE len(parts) <> len(list_distinct(parts)))              AS dup_in_countries,
       count(*) FILTER (WHERE parts <> list_sort(parts))                            AS unsorted_countries,
       count(*) FILTER (WHERE list_sum(list_transform(cc, z -> z[2]::INTEGER)) IS DISTINCT FROM nullif(n_located, 0)) AS counts_sum_mismatch,
       count(*) FILTER (WHERE list_sort(list_transform(cc, z -> z[1])) <> parts)     AS counts_codes_mismatch,
       count(*) FILTER (WHERE first_inventor_country IS NOT NULL AND NOT list_contains(parts, first_inventor_country)) AS first_not_in_list,
       count(*) FILTER (WHERE is_international <> (n_countries > 1))                AS intl_mismatch,
       count(*) FILTER (WHERE countries IS NOT NULL)                                AS with_country,
       count(*) FILTER (WHERE is_international)                                     AS international,
       count(*) FILTER (WHERE assignee_countries IS NOT NULL)                       AS with_assignee_country
FROM x""").fetchdf().iloc[0]
for k, v in chk.items():
    print(f'  {k:<24} {int(v):>14,}')
ok = all(chk[k] == 0 for k in ['dup_patent_id', 'located_gt_inventors', 'n_countries_mismatch', 'dup_in_countries',
                                'unsorted_countries', 'counts_sum_mismatch', 'counts_codes_mismatch',
                                'first_not_in_list', 'intl_mismatch'])
print(f"\n1-3. {'ALL PASS' if ok else 'FAILED -- see the counts above'}")
print(f'     located share: {chk.with_country/chk.patents*100:.1f}% of patents, '
      f'{chk.located_slots/chk.inventor_slots*100:.1f}% of inventor slots; '
      f'international: {chk.international/chk.patents*100:.1f}%')

if os.path.exists(META_FP):
    m = con.execute(f"""
    -- 36 ids contain a ';' of their own (HTML entity 'jadri&cacute;-1', mangled 'ha;feld-1'), so the list is split only
    -- where the next id starts: 'fl:' ids or 20-char hash ids. RE2 has no lookahead, hence the capture-group replace.
    WITH j AS (SELECT n.patent_id, n.n_inventors, len(str_split(regexp_replace(m.inventor_list, ';(fl:|[0-9a-z]{{20}})', '|\\1', 'g'), '|')) AS meta_n
               FROM read_parquet('{OUT_FP}') n JOIN read_parquet('{META_FP}') m USING (patent_id)
               WHERE m.inventor_list IS NOT NULL)
    SELECT count(*) AS shared, count(*) FILTER (WHERE n_inventors <> meta_n) AS n_mismatch FROM j""").fetchdf().iloc[0]
    only_meta = con.execute(f"""SELECT count(*) FROM read_parquet('{META_FP}') m
                               WHERE m.inventor_list IS NOT NULL AND m.patent_id NOT IN (SELECT patent_id FROM read_parquet('{OUT_FP}'))""").fetchone()[0]
    print(f"4.   vs patent_metadata.inventor_list: {int(m.shared):,} shared patents, {int(m.n_mismatch):,} with a different inventor count"
          + ('   OK' if m.n_mismatch == 0 else
             ('   (expected in a smoke test: NB_ROW_LIMIT truncates patents)' if ROW_LIMIT else '   <-- FAILED')))
    print(f'     patents with an inventor_list in patent_metadata but absent here: {only_meta:,}'
          + ('' if not ROW_LIMIT else '   (expected in a smoke test)'))
else:
    print('4.   patent_metadata.parquet not present -- cross-check skipped')

print('\n5.   coverage by grant decade')
con.register('years', years.rename('grant_year').reset_index())
display(con.execute(f"""
SELECT (y.grant_year // 10) * 10 AS decade, count(*) AS patents,
       round(100.0 * count(*) FILTER (WHERE s.countries IS NOT NULL) / count(*), 1) AS pct_located,
       round(100.0 * count(*) FILTER (WHERE s.is_international) / count(*), 2) AS pct_international,
       round(avg(s.n_inventors), 2) AS mean_inventors
FROM read_parquet('{OUT_FP}') s JOIN years y USING (patent_id) GROUP BY 1 ORDER BY 1""").fetchdf())
print('     most frequent inventor countries (patents with >=1 inventor there)')
display(con.execute(f"""
SELECT c AS country, count(*) AS patents, round(100.0*count(*)/(SELECT count(*) FROM read_parquet('{OUT_FP}')), 2) AS pct
FROM read_parquet('{OUT_FP}'), unnest(str_split(countries, ';')) AS u(c)
WHERE countries IS NOT NULL GROUP BY 1 ORDER BY 2 DESC LIMIT 15""").fetchdf())
display(con.execute(f"SELECT * FROM read_parquet('{OUT_FP}') WHERE is_international ORDER BY n_countries DESC LIMIT 5").fetchdf())
con.close()